# State Card Schema Lock

상태카드 기반 외부 검증 산출물을 확인하는 재실행용 노트북입니다.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', 80)

In [2]:
pd.read_csv('09_실험라인/state_card_external_validation/outputs/state_card_output_schema.csv')

,field_name,type,plain_korean_meaning,source,required
0,sample_id,string,상태카드 대상 window 또는 event ID,runtime/window input,True
1,data_scope,string,"M1 내부, M1↔M2, XAI4HEAT 중 출처 구분",validation layer,True
2,substation_id,string,기계실 또는 외부 SCADA 파일 ID,input metadata,True
3,window_start,datetime,센서 feature 계산 window 시작,feature window,False
4,window_end,datetime,센서 feature 계산 window 끝,feature window,False
5,primary_state,string,최종 상태 normal/fault/task/activity/review_required,conflict resolver,True
6,secondary_tags,string,동시에 켜진 보조 gate 목록,conflict resolver,False
7,fault_detected,boolean,fault gate 통과 여부,RandomForest fault gate,True
8,task_detected,boolean,task gate 통과 여부,RandomForest task gate,True
9,activity_detected,boolean,activity gate 통과 여부,RandomForest activity gate,True


In [3]:
pd.read_csv('09_실험라인/state_card_external_validation/outputs/state_card_rule_table.csv')

,rule_id,component,condition,output,model_or_rule,note
0,R1,front_gates,fault/task/activity gate를 병렬 적용,각 gate probability와 detected flag,"RandomForest depth3, threshold 0.5",task는 조기탐지가 아니라 상태/작업 감지
1,R2,conflict_resolver,fault와 task/activity가 동시에 켜짐,"primary_state=fault, secondary_tags에 task/acti...",deterministic rule,운영 위험도가 높은 fault 우선
2,R3,conflict_resolver,task와 activity만 동시에 켜짐,확률이 높은 쪽을 primary_state로 선택,probability comparison,둘 다 secondary tag에도 남김
3,R4,fault_pre_event,primary_state=fault,pre_event_detected와 risk_probability,"pre_event model, threshold 0.6",fault branch에서만 계산
4,R5,leadtime_priority,fault이고 risk_probability >= 0.6,"fault_group, leadtime_label, priority_score, p...",29 stable crossing + 30 baseline policy,priority는 ML 모델이 아니라 정책 점수
5,R6,external_runtime,외부 데이터에 라벨 또는 필수 feature가 없음,review_required 또는 not_available 사유 기록,availability rule,성능점수 대신 coverage/runtime 검증
